# Proyecto Loti Perú — Pipeline MLOps
> Notebooks listos para Databricks. Ajustá la variable `DATA_PATH` para tu ruta.

## 05 · KPI Report (Monitoreo y Drift básico)

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CATALOG = "main"
SCHEMA  = "loterias_silver"
TABLE_FEATURES = f"{CATALOG}.{SCHEMA}.features_apuestas"

pdf = spark.table(TABLE_FEATURES).toPandas()

# KPIs operativos
kpi = {
    "tx_total": int(len(pdf)),
    "fraude_rate": float(pdf["es_fraude"].mean()),
    "monto_promedio": float(pdf["monto"].mean() if "monto" in pdf.columns else np.nan),
    "usuarios_unicos": int(pdf["user_id"].nunique()),
}
kpi

### Curva de distribución de score (si existiera) y drift simple por IP/usuario

In [0]:
# Placeholder de 'score'; si no hay, simular para demo
if "score" not in pdf.columns:
    rng = np.random.RandomState(42)
    pdf["score"] = rng.uniform(0,1,len(pdf)) * (0.3 + 0.7*pdf["es_fraude"].values)

# Histograma de scores
plt.figure()
plt.hist(pdf["score"], bins=30)
plt.title("Distribución de score")
plt.xlabel("score")
plt.ylabel("frecuencia")
plt.show()

In [0]:
# Drift simple: proporción de fraude por IP top
top_ip = (pdf.groupby("ip")["es_fraude"].mean().sort_values(ascending=False).head(10))
top_ip = top_ip.reset_index().rename(columns={"es_fraude":"fraude_rate_ip"})
top_ip